# Madhava 32D->64D Adaptive: Scaling Benchmark vs FlatIP / HNSW(64) / IVF(10/20/50)

**Zero bound violations across all scales - mathematically guaranteed.**

### Pipeline Madhava Adaptive 32D->64D:
1. QR-JL 32D em TODOS N -> Upper bound Cauchy-Schwarz
2. Adaptive keep-ratio: bound range estreito (dados uniformes) = reter ~3.3% | wide (estruturados) = reter mais
3. Refinement 64D nos sobreviventes + error backprop modulation
4. Cosine exato em top-200 -> top-10

**0 violacoes de bound** em todas as escalas (garantia pela ortogonalizacao QR + Pitagoras)

**Ref:** Zenodo 21052709 | Kaggle: https://www.kaggle.com/code/kleniopadilha/madhava-adaptive-32d-64d-scaling-1k-1m
**Licenca:** BSL 1.1 | **Contact:** pay@winnex.ai


In [ ]:
import sys,os,warnings,math,time,random,gc,json
import numpy as np
os.environ['TOKENIZERS_PARALLELISM']='false'; warnings.filterwarnings('ignore')
for pkg,imp in [('faiss-cpu','faiss'),('scikit-learn','sklearn'),('scipy','scipy')]:
    try: __import__(imp)
    except: import subprocess; subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import faiss
SEED=42; np.random.seed(SEED); random.seed(SEED); FINAL_K=10
print('Setup OK')


In [ ]:
import numpy as np, math, time

class MadhavaAdaptive:
    def __init__(self):
        self.full_dim=128; self.base_keep=0.15
        self.max_stage1=150000; self.final_topk=200
        self.rng=np.random.RandomState(43)
        self.vectors=None; self.proj_L={}; self.error={}; self.proj_matrices={}
    def _make_orthogonal_proj(self,d_out,d_in):
        Q,_=np.linalg.qr(self.rng.randn(d_out,d_in).astype(np.float64).T)
        P=Q[:,:d_out].T.astype(np.float32)
        assert np.abs(P@P.T-np.eye(d_out)).max()<1e-5; return P
    def build(self,vectors):
        t0=time.time(); self.vectors=vectors; self.n_vectors=len(vectors)
        self.norms=np.linalg.norm(vectors,axis=1).astype(np.float64)
        for d in [32,64]:
            P=self._make_orthogonal_proj(d,self.full_dim)
            self.proj_matrices[d]=P
            proj=(vectors.astype(np.float32)@P.T).astype(np.float64)
            self.proj_L[d]=proj
            captured=np.linalg.norm(proj,axis=1)
            self.error[d]=np.sqrt(np.maximum(self.norms**2-captured**2,0))
        self.build_time=time.time()-t0; return self
    def _upper_bound(self,pv,ev,pq,eq): return pv@pq+ev*eq
    def search(self,q,retprof=False):
        q=q.astype(np.float64).flatten(); qn=np.linalg.norm(q); p={'n_total':self.n_vectors}
        q32=(q.astype(np.float32)@self.proj_matrices[32].T).astype(np.float64)
        qr32=math.sqrt(max(0,qn**2-np.linalg.norm(q32)**2))
        B32=self._upper_bound(self.proj_L[32],self.error[32],q32,qr32)
        b_range=float(B32.max()-B32.min())
        adaptive_keep=min(0.30,max(0.02,self.base_keep*0.1/max(b_range,0.01)))
        k1=min(max(int(self.n_vectors*adaptive_keep),500),self.max_stage1,self.n_vectors)
        if self.n_vectors<=k1: idx1=np.arange(self.n_vectors)
        else: idx1=np.argpartition(-B32,k1-1)[:k1]
        p['n1']=len(idx1); p['adaptive_keep']=adaptive_keep
        q64=(q.astype(np.float32)@self.proj_matrices[64].T).astype(np.float64)
        qr64=math.sqrt(max(0,qn**2-np.linalg.norm(q64)**2))
        B64=self._upper_bound(self.proj_L[64][idx1],self.error[64][idx1],q64,qr64)
        e32=self.error[32][idx1]; e64=self.error[64][idx1]
        al=1.0/(1.0+np.exp(-(e32-e64)/max(np.mean(e32),1e-9)*0.5))
        p['alpha']=float(np.mean(al))
        mod=B32[idx1]+al*(B64-B32[idx1])
        k2=min(self.final_topk,len(idx1))
        idx2=idx1[np.argpartition(-mod,k2-1)[:k2]]
        top=idx2[np.argsort(-(self.vectors[idx2].astype(np.float32)@q.astype(np.float32)))][:FINAL_K]
        if retprof: return top,p
        return top
    def bound_violations(self,q):
        q=q.astype(np.float64).flatten(); qn=np.linalg.norm(q)
        tc=self.vectors.astype(np.float64)@q; v={}
        for d in [32,64]:
            qd=(q.astype(np.float32)@self.proj_matrices[d].T).astype(np.float64)
            ub=self._upper_bound(self.proj_L[d],self.error[d],qd,math.sqrt(max(0,qn**2-np.linalg.norm(qd)**2)))
            v[f'{d}D']=int(np.sum(tc>ub+1e-9))
        return v,self.n_vectors


In [ ]:
def make_s(n,dim,s=SEED):
    rng=np.random.RandomState(s); X=rng.randn(n,dim).astype(np.float32)
    X/=np.linalg.norm(X,axis=1,keepdims=True); return X


In [ ]:
print('='*85)
print('  MADHAVA ADAPTIVE 32D->64D vs FlatIP / HNSW / IVF: ESCALABILIDADE 1K A 1M')
print('='*85)
all_res={}
Ns=[1000,5000,10000,50000,200000,500000,1000000]
nq_map={1000:15,5000:10,10000:7,50000:5,200000:3,500000:2,1000000:2}

print(f"{'N':>10} {'FlatIP':>8} {'HNSW64':>9} {'IVF10':>9} {'IVF20':>9} {'IVF50':>9} {'Mad32D':>9} {'Build':>7} {'NDCG':>7} {'Viol':>5}")
print(f"{'-'*10} {'-'*8} {'-'*9} {'-'*9} {'-'*9} {'-'*9} {'-'*9} {'-'*7} {'-'*7} {'-'*5}")

for nc in Ns:
    nq=nq_map[nc]
    E=make_s(nc,128,SEED+nc); Q=make_s(nq,128,SEED+9999+nc)

    # FlatIP
    fi=faiss.IndexFlatIP(128); fi.add(E)
    ft=[(time.time(),fi.search(Q[qi:qi+1],FINAL_K)) for qi in range(nq)]
    flat_ms=np.mean([(time.time()-t0)*1000 for t0,_ in ft])

    # HNSW(64)
    idx0=faiss.IndexHNSWFlat(128,32); idx0.hnsw.efConstruction=200; idx0.add(E); idx0.hnsw.efSearch=64
    ht=[(time.time(),idx0.search(Q[qi:qi+1],FINAL_K)) for qi in range(nq)]
    hnsw_ms=np.mean([(time.time()-t0)*1000 for t0,_ in ht])

    # IVF(nprobe=10,20,50)
    nlist=min(int(math.sqrt(nc)),256)
    ivf_ms={}
    for npb in [10,20,50]:
        quant=faiss.IndexFlatIP(128); idx1=faiss.IndexIVFFlat(quant,128,nlist,faiss.METRIC_INNER_PRODUCT)
        idx1.train(E); idx1.add(E); idx1.nprobe=npb
        it=[(time.time(),idx1.search(Q[qi:qi+1],FINAL_K)) for qi in range(nq)]
        ivf_ms[npb]=float(np.mean([(time.time()-t0)*1000 for t0,_ in it]))

    # Madhava Adaptive 32D->64D
    mad=MadhavaAdaptive(); mad.build(E)
    mt,ndcg_l,p_list=[],[],[]
    for qi in range(nq):
        qv=Q[qi]; ts={int(i):float(E[i]@qv) for i in np.argsort(-(E@qv))[:FINAL_K]}
        t0=time.time(); top,p=mad.search(qv,retprof=True); mt.append((time.time()-t0)*1000); p_list.append(p)
        dcg=sum((2**ts.get(int(idx),0)-1)/math.log2(j+2) for j,idx in enumerate(top[:FINAL_K]))
        idcg=sum((2**v-1)/math.log2(j+2) for j,(_,v) in enumerate(sorted(ts.items(),key=lambda x:x[1],reverse=True)[:FINAL_K]))
        ndcg_l.append(dcg/idcg if idcg>0 else 0)

    mad_ms=np.mean(mt)
    viols={}
    for qi in range(min(3,nq)):
        v,_=mad.bound_violations(Q[qi])
        for kk,vv in v.items(): viols[kk]=max(viols.get(kk,0),vv)
    vio_flag='OK' if all(v==0 for v in viols.values()) else '!'

    print(f"{nc:>10,} {flat_ms:>7.2f}ms {hnsw_ms:>8.2f}ms {ivf_ms[10]:>8.2f}ms {ivf_ms[20]:>8.2f}ms {ivf_ms[50]:>8.2f}ms {mad_ms:>8.2f}ms {mad.build_time:>6.3f}s {np.mean(ndcg_l):>7.4f} {vio_flag:>5}")
    all_res[f'N={nc}']={'flat_ms':float(flat_ms),'hnsw_ms':float(hnsw_ms),'ivf_10':float(ivf_ms[10]),'ivf_20':float(ivf_ms[20]),'ivf_50':float(ivf_ms[50]),'mad_ms':float(mad_ms),'build_s':float(mad.build_time),'ndcg':float(np.mean(ndcg_l)),'n1':int(np.mean([p['n1'] for p in p_list])),'viol':viols}
    del E,Q,mad,fi,idx0; gc.collect()

with open('madhava_adaptive_results.json','w') as f: json.dump(all_res,f,indent=2,ensure_ascii=False)

print()
print('='*85)
print('  TABELA FINAL')
print('='*85)
print(f"{'N':>10} {'FlatIP':>8} {'HNSW64':>9} {'IVF10':>9} {'IVF20':>9} {'IVF50':>9} {'Mad32D':>9} {'Build':>7} {'NDCG':>7} {'Viol':>5}")
print(f"{'-'*10} {'-'*8} {'-'*9} {'-'*9} {'-'*9} {'-'*9} {'-'*9} {'-'*7} {'-'*7} {'-'*5}")
for nc in Ns:
    r=all_res.get(f'N={nc}',{})
    vio='OK' if all(v==0 for v in r.get('viol',{}).values()) else '!'
    print(f"{nc:>10,} {r.get('flat_ms',0):>7.2f}ms {r.get('hnsw_ms',0):>8.2f}ms {r.get('ivf_10',0):>8.2f}ms {r.get('ivf_20',0):>8.2f}ms {r.get('ivf_50',0):>8.2f}ms {r.get('mad_ms',0):>8.2f}ms {r.get('build_s',0):>6.3f}s {r.get('ndcg',0):>7.4f} {vio:>5}")
print('\nOK. Zenodo: 10.5281/zenodo.21052709 | BSL 1.1')